In [1]:
# Cell 1
import pandas as pd
import numpy as np

df = pd.read_csv('simulados_pairs_features.csv')
print(df.shape)
df.head()

(115, 19)


,aluno_id,prev_created_at,prev_matematica,prev_linguagens,prev_humanas,prev_natureza,next_matematica,next_linguagens,next_humanas,next_natureza,tests_count,trend_matematica,trend_linguagens,trend_humanas,trend_natureza,rolling_mean_matematica,rolling_mean_linguagens,rolling_mean_humanas,rolling_mean_natureza
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-02 18:01:44.226071+00:00,25.166667,29.833333,31.833333,25.0,25.0,NaN,NaN,21.0,1,NaN,NaN,NaN,NaN,25.166667,29.833333,31.833333,25.0
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-28 01:32:38.720997+00:00,25.000000,NaN,NaN,21.0,24.0,NaN,NaN,NaN,2,-0.166667,NaN,NaN,-4.000000e+00,25.083333,29.833333,31.833333,23.0
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-05-06 20:37:03.468013+00:00,24.000000,NaN,NaN,NaN,24.0,27.0,33.0,25.0,3,-0.583333,NaN,NaN,-4.000000e+00,24.722222,29.833333,31.833333,23.0
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:10:21.903540+00:00,24.000000,27.000000,33.000000,25.0,31.0,23.0,38.0,25.0,4,-0.450000,-2.833333,1.166667,-1.522248e-15,24.333333,27.000000,33.000000,23.0
4,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-02 18:23:16.513865+00:00,41.000000,40.000000,40.000000,39.0,35.0,33.0,35.0,31.0,1,NaN,NaN,NaN,NaN,41.000000,40.000000,40.000000,39.0


In [2]:
# Cell 2 — check how many NaNs exist per column
df.isnull().sum()

aluno_id                    0
prev_created_at             0
prev_matematica            12
prev_linguagens             8
prev_humanas                7
prev_natureza              13
next_matematica            16
next_linguagens            10
next_humanas                9
next_natureza              17
tests_count                 0
trend_matematica           27
trend_linguagens           27
trend_humanas              27
trend_natureza             27
rolling_mean_matematica     2
rolling_mean_linguagens     0
rolling_mean_humanas        0
rolling_mean_natureza       2
dtype: int64

In [3]:
# Cell 3 — fill NaNs in feature columns

# trend: 0 means "no directional signal available"
trend_cols = [f'trend_{area}' for area in ['matematica', 'linguagens', 'humanas', 'natureza']]
df[trend_cols] = df[trend_cols].fillna(0)

# rolling mean: fill with each column's mean across all students
rolling_cols = [f'rolling_mean_{area}' for area in ['matematica', 'linguagens', 'humanas', 'natureza']]
df[rolling_cols] = df[rolling_cols].fillna(df[rolling_cols].mean())

# verify
df.isnull().sum()

aluno_id                    0
prev_created_at             0
prev_matematica            12
prev_linguagens             8
prev_humanas                7
prev_natureza              13
next_matematica            16
next_linguagens            10
next_humanas                9
next_natureza              17
tests_count                 0
trend_matematica            0
trend_linguagens            0
trend_humanas               0
trend_natureza              0
rolling_mean_matematica     0
rolling_mean_linguagens     0
rolling_mean_humanas        0
rolling_mean_natureza       0
dtype: int64

In [4]:
# Cell 4 — build the matematica model

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np

area = 'matematica'

# step 1 — filter to rows where both input and target exist for this area
mask = df['prev_matematica'].notna() & df['next_matematica'].notna()
area_df = df[mask].copy()
print(f"Rows available for {area} model: {len(area_df)}")

# step 2 — define X and Y
# X: the features the model can see (everything except next_* scores and identifiers)
feature_cols = [
    f'prev_{area}',
    f'trend_{area}',
    f'rolling_mean_{area}',
    'tests_count'
]
X = area_df[feature_cols]
y = area_df[f'next_{area}']

# step 3 — leave-one-student-out cross validation
students = area_df['aluno_id'].unique()
errors = []  # will store the prediction error for each student

for student in students:
    # training set: every student EXCEPT this one
    train_mask = area_df['aluno_id'] != student
    X_train = X[train_mask]
    y_train = y[train_mask]

    # test set: just this student
    X_test = X[~train_mask]
    y_test = y[~train_mask]

    # skip if this student has no training data or only 1 test
    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # mean absolute error: average of |predicted - actual|, in points (0-45 scale)
    mae = mean_absolute_error(y_test, predictions)
    errors.append(mae)

print(f"Leave-one-student-out MAE per student: {[round(e,1) for e in errors]}")
print(f"Average MAE: {round(np.mean(errors), 2)} points")
print(f"Worst MAE: {round(np.max(errors), 2)} points")

Rows available for matematica model: 90
Leave-one-student-out MAE per student: [3.9, 6.9, 3.0, 2.9, 11.2, 2.9, 1.7, 2.9, 10.5, 2.6, 4.5, 0.4, 1.2, 5.7, 2.5, 2.1, 1.2, 1.5, 1.6, 2.7, 2.8, 1.0, 4.7]
Average MAE: 3.49 points
Worst MAE: 11.25 points


In [5]:
# Cell 5 — find which students had the highest error and inspect their data

students = area_df['aluno_id'].unique()
student_errors = []

for student in students:
    train_mask = area_df['aluno_id'] != student
    X_train = X[train_mask]
    y_train = y[train_mask]
    X_test = X[~train_mask]
    y_test = y[~train_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    student_errors.append({'aluno_id': student, 'mae': round(mae, 2), 'n_pairs': len(X_test)})

error_df = pd.DataFrame(student_errors).sort_values('mae', ascending=False)
error_df.head(10)

,aluno_id,mae,n_pairs
4,2d95dc73-c750-4806-a7cf-0deb07faad30,11.25,3
8,4e8115b5-3e5b-439c-867b-a876befcdea4,10.46,1
1,1685cd68-0db4-41e2-84f7-5065022bb134,6.91,4
13,5b152cab-3b5e-43c2-8491-f5c88ef7d9f5,5.68,3
22,e747b89c-ae5b-440f-ab3f-05088af5b560,4.66,3
10,58d56284-c5de-4e5a-beff-b81975b6c09a,4.47,4
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,3.90,4
2,2119aaa5-f0a9-4cb8-bc30-3a5504efd77d,2.98,4
3,233afb42-cb42-41fe-a7fc-e865bc9615f1,2.91,1
7,4d3ee5f1-21e1-444d-94fd-d3d0832f0ea6,2.90,1


In [6]:
# Cell 6 — inspect the weights the model learned
# train on ALL data this time (no holdout) to see the full model's coefficients

model_full = LinearRegression()
model_full.fit(X, y)

weights = pd.DataFrame({
    'feature': feature_cols,
    'weight': model_full.coef_
}).sort_values('weight', ascending=False)

print(f"Intercept: {round(model_full.intercept_, 2)}")
print(weights.to_string(index=False))

Intercept: 9.86
                feature    weight
rolling_mean_matematica  0.791407
            tests_count  0.227180
        prev_matematica -0.076124
       trend_matematica -0.333068


In [7]:
# Cell 7 — build the linguagens model

area = 'linguagens'

# step 1 — filter to rows where both input and target exist for this area
mask = df['prev_linguagens'].notna() & df['next_linguagens'].notna()
area_df = df[mask].copy()
print(f"Rows available for {area} model: {len(area_df)}")

# step 2 — define X and Y
# X: the features the model can see (everything except next_* scores and identifiers)
feature_cols = [
    f'prev_{area}',
    f'trend_{area}',
    f'rolling_mean_{area}',
    'tests_count'
]
X = area_df[feature_cols]
y = area_df[f'next_{area}']

# step 3 — leave-one-student-out cross validation
students = area_df['aluno_id'].unique()
errors = []  # will store the prediction error for each student

for student in students:
    # training set: every student EXCEPT this one
    train_mask = area_df['aluno_id'] != student
    X_train = X[train_mask]
    y_train = y[train_mask]

    # test set: just this student
    X_test = X[~train_mask]
    y_test = y[~train_mask]

    # skip if this student has no training data or only 1 test
    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # mean absolute error: average of |predicted - actual|, in points (0-45 scale)
    mae = mean_absolute_error(y_test, predictions)
    errors.append(mae)

print(f"Leave-one-student-out MAE per student: {[round(e,1) for e in errors]}")
print(f"Average MAE: {round(np.mean(errors), 2)} points")
print(f"Worst MAE: {round(np.max(errors), 2)} points")

Rows available for linguagens model: 98
Leave-one-student-out MAE per student: [8.0, 5.6, 1.5, 0.1, 2.4, 9.6, 3.7, 1.7, 0.7, 2.1, 2.3, 1.1, 3.3, 2.3, 3.1, 2.6, 4.1, 3.3, 2.1, 1.9, 1.8, 4.3, 3.5, 3.6]
Average MAE: 3.12 points
Worst MAE: 9.61 points


In [8]:
# Cell 8 — build the humanas model

area = 'humanas'

# step 1 — filter to rows where both input and target exist for this area
mask = df['prev_humanas'].notna() & df['next_humanas'].notna()
area_df = df[mask].copy()
print(f"Rows available for {area} model: {len(area_df)}")

# step 2 — define X and Y
# X: the features the model can see (everything except next_* scores and identifiers)
feature_cols = [
    f'prev_{area}',
    f'trend_{area}',
    f'rolling_mean_{area}',
    'tests_count'
]
X = area_df[feature_cols]
y = area_df[f'next_{area}']

# step 3 — leave-one-student-out cross validation
students = area_df['aluno_id'].unique()
errors = []  # will store the prediction error for each student

for student in students:
    # training set: every student EXCEPT this one
    train_mask = area_df['aluno_id'] != student
    X_train = X[train_mask]
    y_train = y[train_mask]

    # test set: just this student
    X_test = X[~train_mask]
    y_test = y[~train_mask]

    # skip if this student has no training data or only 1 test
    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # mean absolute error: average of |predicted - actual|, in points (0-45 scale)
    mae = mean_absolute_error(y_test, predictions)
    errors.append(mae)

print(f"Leave-one-student-out MAE per student: {[round(e,1) for e in errors]}")
print(f"Average MAE: {round(np.mean(errors), 2)} points")
print(f"Worst MAE: {round(np.max(errors), 2)} points")

Rows available for humanas model: 100
Leave-one-student-out MAE per student: [2.8, 2.6, 2.9, 5.4, 6.9, 12.9, 3.2, 4.2, 5.9, 4.8, 2.8, 1.2, 3.4, 2.2, 7.3, 2.3, 1.8, 3.6, 2.7, 2.7, 1.2, 2.1, 2.3, 2.7]
Average MAE: 3.75 points
Worst MAE: 12.93 points


In [9]:
# Cell 9 — build the natureza model

area = 'natureza'

# step 1 — filter to rows where both input and target exist for this area
mask = df['prev_natureza'].notna() & df['next_natureza'].notna()
area_df = df[mask].copy()
print(f"Rows available for {area} model: {len(area_df)}")

# step 2 — define X and Y
# X: the features the model can see (everything except next_* scores and identifiers)
feature_cols = [
    f'prev_{area}',
    f'trend_{area}',
    f'rolling_mean_{area}',
    'tests_count'
]
X = area_df[feature_cols]
y = area_df[f'next_{area}']

# step 3 — leave-one-student-out cross validation
students = area_df['aluno_id'].unique()
errors = []  # will store the prediction error for each student

for student in students:
    # training set: every student EXCEPT this one
    train_mask = area_df['aluno_id'] != student
    X_train = X[train_mask]
    y_train = y[train_mask]

    # test set: just this student
    X_test = X[~train_mask]
    y_test = y[~train_mask]

    # skip if this student has no training data or only 1 test
    if len(X_train) == 0 or len(X_test) == 0:
        continue

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # mean absolute error: average of |predicted - actual|, in points (0-45 scale)
    mae = mean_absolute_error(y_test, predictions)
    errors.append(mae)

print(f"Leave-one-student-out MAE per student: {[round(e,1) for e in errors]}")
print(f"Average MAE: {round(np.mean(errors), 2)} points")
print(f"Worst MAE: {round(np.max(errors), 2)} points")

Rows available for natureza model: 86
Leave-one-student-out MAE per student: [3.0, 11.7, 2.2, 5.8, 2.0, 5.3, 2.5, 5.4, 4.7, 4.9, 6.0, 8.4, 4.1, 6.2, 1.6, 5.5, 0.0, 3.4, 1.6, 3.1, 2.7, 3.2, 4.6]
Average MAE: 4.26 points
Worst MAE: 11.66 points
